# MLP 手写数字识别：API 预备知识

这个 notebook 先不急着完整训练 MNIST 模型，而是先把项目里会反复出现的 PyTorch API 搞清楚。

我们最终要做的事情是：输入一张 `28 x 28` 的手写数字图片，让 MLP 判断它属于 `0 ~ 9` 中的哪一个数字。

完整训练流程可以先记成这一句话：

```text
图片 -> 模型 forward -> logits -> CrossEntropyLoss -> backward -> optimizer.step -> accuracy
```

本节会依次学习：

- `class Model(nn.Module)`
- `forward()`
- `CrossEntropyLoss`
- `optimizer`
- train loop
- test / evaluate loop
- accuracy 计算


## 0. 先导入会用到的库

下面这些库是后续 MNIST 项目里最常见的组合。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print(torch.__version__)

2.10.0


## 1. `class Model(nn.Module)` 是什么

`nn.Module` 是 PyTorch 中所有神经网络模型的基类。

自己写模型时，通常会写成：

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...
```

`nn.Module` 的核心作用：

- 管理模型参数，例如 `model.parameters()`。
- 管理子层，例如 `self.fc1 = nn.Linear(...)`。全连接层就这么写就行了。
- 支持把模型移动到 CPU/GPU，例如 `model.to(device)`。
- 支持训练/评估模式切换，例如 `model.train()` 和 `model.eval()`。
- 支持保存和加载参数，例如 `model.state_dict()`。

在 MNIST 的 MLP 项目里，我们通常会定义这些结构参数：

| 参数 | 含义 | MNIST 中的常见值 |
| --- | --- | --- |
| `input_size` | 输入特征数 | `28 * 28 = 784` |
| `hidden_size` | 隐藏层神经元数量 | `128`、`256` 等 |
| `num_classes` | 输出类别数 | `10` |


In [3]:
class MLP(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = MLP()
model

MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

## 2. `forward()` 是什么

`forward()` 定义模型的前向传播过程，也就是输入数据如何一步一步变成输出结果。

对 MNIST MLP 来说，输入图片通常是：

```text
[batch_size, 1, 28, 28]
```

但是全连接层 `nn.Linear` 需要二维输入：

```text
[batch_size, feature_size]
```

所以第一步通常是展平：

```python
x = x.view(x.size(0), -1)
```

`forward(self, x)` 的参数含义：

| 参数 | 含义 |
| --- | --- |
| `self` | 当前模型对象 |
| `x` | 输入数据，比如一批 MNIST 图片 |
| 返回值 | 模型输出，通常叫 `logits` |

注意：平时调用模型时写 `model(images)`，而不是直接写 `model.forward(images)`。`model(images)` 会自动触发 PyTorch 模型调用机制。

In [4]:
# 模拟一个 batch：4 张灰度图片，每张 28 x 28
fake_images = torch.randn(4, 1, 28, 28)

logits = model(fake_images)

print("input shape:", fake_images.shape)
print("output/logits shape:", logits.shape)

input shape: torch.Size([4, 1, 28, 28])
output/logits shape: torch.Size([4, 10])


## 3. `CrossEntropyLoss` 是什么

`CrossEntropyLoss` 是多分类任务中最常用的损失函数。

MNIST 是 10 分类任务，所以很适合用它：

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
```

它衡量的是：模型给出的分类分数和真实标签之间差得有多远。

重要：传给 `CrossEntropyLoss` 的应该是原始 `logits`，不要提前做 `Softmax`。因为 `CrossEntropyLoss` 内部已经包含了类似 `LogSoftmax + NLLLoss` 的计算。

常用参数：

```python
nn.CrossEntropyLoss(
    weight=None,
    size_average=None,
    ignore_index=-100,
    reduce=None,
    reduction='mean',
    label_smoothing=0.0
)
```

| 参数 | 含义 | MNIST 中怎么用 |
| --- | --- | --- |
| `weight` | 给不同类别设置不同损失权重 | 类别不均衡时使用，MNIST 通常 `None` |
| `size_average` | 旧参数，控制是否平均 | 已不推荐，使用 `reduction` |
| `ignore_index` | 忽略某个标签，不参与 loss | MNIST 通常不用 |
| `reduce` | 旧参数，控制是否汇总 loss | 已不推荐，使用 `reduction` |
| `reduction` | loss 汇总方式：`'mean'`、`'sum'`、`'none'` | 通常用默认 `'mean'` |
| `label_smoothing` | 标签平滑，减少模型过度自信 | 入门先用 `0.0` |

输入输出形状：

| 数据 | 形状 | 说明 |
| --- | --- | --- |
| `logits` | `[batch_size, 10]` | 每个样本对 10 类的原始分数 |
| `labels` | `[batch_size]` | 每个样本的真实类别编号 |
| `loss` | 标量 | 当前 batch 的平均损失 |


In [ ]:
criterion = nn.CrossEntropyLoss()

# 假设 4 张图片的真实标签分别是 7、2、1、0
fake_labels = torch.tensor([7, 2, 1, 0])

loss = criterion(logits, fake_labels)

print("logits shape:", logits.shape)
print("labels shape:", fake_labels.shape)
print("loss:", loss.item())

## 4. `optimizer` 是什么

优化器负责根据梯度更新模型参数。

反向传播 `loss.backward()` 会计算梯度，优化器 `optimizer.step()` 会真正修改参数。

常见写法：

```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

通用核心参数：

| 参数 | 含义 |
| --- | --- |
| `params` | 要优化的参数，通常传 `model.parameters()` |
| `lr` | learning rate，学习率，控制每次更新的步子大小 |

### SGD

```python
torch.optim.SGD(
    params,
    lr=0.001,
    momentum=0,
    dampening=0,
    weight_decay=0,
    nesterov=False
)
```

| 参数 | 含义 |
| --- | --- |
| `params` | 要更新的模型参数 |
| `lr` | 学习率 |
| `momentum` | 动量，参考过去的更新方向，常见值 `0.9` |
| `dampening` | 动量阻尼，入门基本不用改 |
| `weight_decay` | 权重衰减，也就是 L2 正则化 |
| `nesterov` | 是否使用 Nesterov 动量 |

### Adam

```python
torch.optim.Adam(
    params,
    lr=0.001,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0
)
```

| 参数 | 含义 |
| --- | --- |
| `params` | 要更新的模型参数 |
| `lr` | 学习率 |
| `betas` | 一阶动量和二阶动量的衰减系数 |
| `eps` | 防止除以 0 的小常数 |
| `weight_decay` | 权重衰减 |

MNIST MLP 入门项目建议先用 Adam：

```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer

## 5. train loop：训练循环

训练循环的标准顺序非常重要：

```text
1. model.train()
2. 取一个 batch 的 images 和 labels
3. optimizer.zero_grad()
4. outputs = model(images)
5. loss = criterion(outputs, labels)
6. loss.backward()
7. optimizer.step()
```

每一步的作用：

| 代码 | 作用 |
| --- | --- |
| `model.train()` | 切换到训练模式 |
| `optimizer.zero_grad()` | 清空上一轮梯度 |
| `outputs = model(images)` | 前向传播，得到预测分数 |
| `loss = criterion(outputs, labels)` | 计算损失 |
| `loss.backward()` | 反向传播，计算梯度 |
| `optimizer.step()` | 根据梯度更新参数 |

为什么要 `zero_grad()`？

PyTorch 默认会累积梯度。如果不清空，当前 batch 的梯度会和上一个 batch 的梯度加在一起，训练结果就会不符合预期。

In [ ]:
# 这里先用假数据演示一个 batch 的训练步骤
model.train()

images = torch.randn(32, 1, 28, 28)
labels = torch.randint(0, 10, (32,))

optimizer.zero_grad()
outputs = model(images)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()

print("train loss:", loss.item())

## 6. test / evaluate loop：测试或评估循环

测试循环只负责评估模型效果，不更新参数。

标准顺序：

```text
1. model.eval()
2. with torch.no_grad()
3. outputs = model(images)
4. loss = criterion(outputs, labels)
5. preds = outputs.argmax(dim=1)
6. 统计 correct 和 total
```

| 代码 | 作用 |
| --- | --- |
| `model.eval()` | 切换到评估模式 |
| `torch.no_grad()` | 关闭梯度计算，节省内存和计算量 |
| `outputs = model(images)` | 得到预测分数 |
| `criterion(outputs, labels)` | 计算测试损失 |
| `argmax(dim=1)` | 从 10 类分数中取最大分数对应的类别 |

测试阶段不需要：

- `optimizer.zero_grad()`
- `loss.backward()`
- `optimizer.step()`

In [ ]:
# 用假数据演示一个 batch 的评估步骤
model.eval()

images = torch.randn(32, 1, 28, 28)
labels = torch.randint(0, 10, (32,))

with torch.no_grad():
    outputs = model(images)
    loss = criterion(outputs, labels)
    preds = outputs.argmax(dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total

print("test loss:", loss.item())
print("correct:", correct)
print("total:", total)
print("accuracy:", accuracy)

## 7. accuracy 准确率怎么计算

准确率公式：

$$
accuracy = \frac{预测正确的样本数}{总样本数}
$$

代码通常写成：

```python
preds = outputs.argmax(dim=1)
correct += (preds == labels).sum().item()
total += labels.size(0)
accuracy = correct / total
```

逐行解释：

| 代码 | 含义 |
| --- | --- |
| `outputs.argmax(dim=1)` | 找到每个样本分数最高的类别 |
| `preds == labels` | 判断预测类别是否等于真实标签 |
| `.sum().item()` | 统计预测正确的数量，并转成 Python 数字 |
| `labels.size(0)` | 当前 batch 的样本数量 |
| `correct / total` | 准确率 |

如果 `accuracy = 0.965`，就表示准确率是 `96.5%`。

## 8. 把所有 API 串起来

真正写 MNIST 项目时，你会看到下面这种整体结构：

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct / total
    return avg_loss, accuracy

## 9. 小结

| API / 概念 | 一句话理解 |
| --- | --- |
| `nn.Module` | 模型的基类，负责管理网络层和参数 |
| `forward()` | 定义输入如何变成输出 |
| `CrossEntropyLoss` | 多分类损失函数，比较 logits 和真实类别 |
| `optimizer` | 根据梯度更新模型参数 |
| `optimizer.zero_grad()` | 清空旧梯度 |
| `loss.backward()` | 计算梯度 |
| `optimizer.step()` | 更新参数 |
| `model.train()` | 开启训练模式 |
| `model.eval()` | 开启评估模式 |
| `torch.no_grad()` | 评估时关闭梯度计算 |
| `argmax(dim=1)` | 从类别分数中选出预测类别 |
| `accuracy` | 预测正确数量除以总数量 |

下一步就可以正式创建 MNIST 数据加载、MLP 模型、训练和测试完整项目了。